# Gradient Descent (Animated)

Visualize how the regression line fits the data as gradient descent updates `m` and `b` over each iteration.

In [4]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

In [5]:
# Dataset
X = np.array([1, 2, 3, 4])
Y = np.array([2, 4, 6, 8])

# Center the features so the loss surface is well-conditioned.
# This decouples slope and intercept, letting both converge quickly.
X_mean = X.mean()
Xc = X - X_mean

# Gradient Descent Settings (work in centered space)
m = 0.0   # slope (same in centered and original space)
b = 0.1   # intercept in CENTERED space
lr = 0.1  # centering decouples the params, so a larger step is safe
epochs = 50
n = len(X)

# Store parameters for every iteration
history = []

# Total variance of Y (denominator for the R^2 / accuracy score)
ss_total = np.sum((Y - np.mean(Y)) ** 2)

for epoch in range(epochs + 1):

    y_pred = m * Xc + b

    # Loss (Mean Squared Error)
    mse = np.mean((Y - y_pred) ** 2)

    # Accuracy (R^2 score: 1.0 == perfect fit)
    ss_res = np.sum((Y - y_pred) ** 2)
    accuracy = 1 - ss_res / ss_total

    # Convert the intercept back to the ORIGINAL X space for plotting:
    # y = m*(X - X_mean) + b  ->  y = m*X + (b - m*X_mean)
    b_real = b - m * X_mean

    history.append({
        "epoch": epoch,
        "m": m,
        "b": b_real,
        "loss": mse,
        "mse": mse,
        "accuracy": accuracy
    })

    # Gradients (computed on centered features)
    dm = (-2 / n) * np.sum(Xc * (Y - y_pred))
    db = (-2 / n) * np.sum(Y - y_pred)

    # Update
    m -= lr * dm
    b -= lr * db

In [7]:
# -------------------------
# Animation
# -------------------------

fig, (ax, ax_loss) = plt.subplots(1, 2, figsize=(14, 6))

# --- Left: regression line fitting the data ---
ax.scatter(X, Y, s=100, label="Actual Data")

line, = ax.plot([], [], lw=2, label="Regression Line")

text = ax.text(
    0.05,
    0.95,
    "",
    transform=ax.transAxes,
    va="top",
    fontsize=12,
    bbox=dict(facecolor="white", alpha=0.8)
)

ax.set_xlim(0, 5)
ax.set_ylim(0, 10)
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_title("Linear Regression: Gradient Descent")
ax.grid(True)
ax.legend()

# --- Right: loss (MSE) curve over iterations ---
epochs_all = [h["epoch"] for h in history]
mse_all = [h["mse"] for h in history]

loss_line, = ax_loss.plot([], [], lw=2, color="crimson", label="MSE")
loss_point, = ax_loss.plot([], [], "o", color="crimson")

ax_loss.set_xlim(0, max(epochs_all))
ax_loss.set_ylim(0, max(mse_all) * 1.05)
ax_loss.set_xlabel("Iteration")
ax_loss.set_ylabel("MSE")
ax_loss.set_title("Loss over Iterations")
ax_loss.grid(True)
ax_loss.legend()


def update(frame):

    h = history[frame]

    m = h["m"]
    b = h["b"]

    x_line = np.array([0, 5])
    y_line = m * x_line + b

    line.set_data(x_line, y_line)

    text.set_text(
        f"Iteration {h['epoch']}\n"
        f"y = {m:.3f}x + {b:.3f}\n"
        f"MSE = {h['mse']:.4f}"
    )

    # Grow the loss curve up to the current frame
    loss_line.set_data(epochs_all[:frame + 1], mse_all[:frame + 1])
    loss_point.set_data([epochs_all[frame]], [mse_all[frame]])

    return line, text, loss_line, loss_point


ani = FuncAnimation(
    fig,
    update,
    frames=len(history),
    interval=200,   # 1 second per frame
    repeat=False
)

# Render the animation inline (plt.show() alone won't play it in Jupyter)
plt.close(fig)  # prevent a duplicate static figure from displaying
HTML(ani.to_jshtml())